# Evaluation of a selection of models on 60km -> 2.2km-4x over Birmingham

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv


In [ ]:
from mlde_analysis.default_params import *

In [ ]:
import functools
from importlib.resources import files
import math
import os
import string

import cftime
import iris
import iris.analysis.cartography
import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pysteps
import scipy
import xarray as xr

import mlde_utils
from mlde_utils import cp_model_rotated_pole, TIME_PERIODS, platecarree, DatasetMetadata
from mlde_analysis.data import prep_eval_data, open_concat_sample_datasets, si_to_mmday
from mlde_analysis import create_map_fig, STYLES
from mlde_analysis.distribution import normalized_mean_bias, normalized_std_bias, plot_freq_density, plot_biases

In [ ]:
xr.set_options(display_style="html")
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
precomp_60km_ds = open_concat_sample_datasets(sample_configs_at_60km, split, ensemble_members=ensemble_members, samples_per_run=samples_per_run)
precomp_60km_ds

In [ ]:
raw_gcm_pr = si_to_mmday(xr.open_dataset(DatasetMetadata(dataset_configs_at_60km["GCM"]).split_path(split))["pr"]).sel(ensemble_member=ensemble_members)
raw_gcm_pr

In [ ]:
cpm_pr_on_gcm = si_to_mmday(xr.open_dataset(DatasetMetadata(dataset_configs_at_60km["CPM"]).split_path(split))["pr"]).sel(ensemble_member=ensemble_members)
cpm_pr_on_gcm

### frequency distribution on the coarse grid, bias in mean and std. dev.

In [ ]:
hist_data = [
    dict(data=precomp_60km_ds["pred_pr"].sel(model=c["label"]), label=c["label"], color=c.get("color", "tab:green"))
    for c in sample_configs_at_60km
] + [dict(data=raw_gcm_pr, label="GCM", color="tab:purple")]

mean_biases = [ dict(data=normalized_mean_bias(hd["data"], cpm_pr_on_gcm), label=hd["label"]) for hd in hist_data ]

std_biases = [ dict(data=normalized_std_bias(hd["data"], cpm_pr_on_gcm), label=hd["label"]) for hd in hist_data ]

In [ ]:
fig = plt.figure(layout='constrained', figsize=(5.5, 6.5))

meanb_axes_keys = list([f"meanb {mb['label']}" for mb in mean_biases])
meanb_spec = np.array(meanb_axes_keys).reshape(1,-1)

stddevb_axes_keys = list([f"stddevb {sb['label']}" for sb in std_biases])
stddevb_spec = np.array(stddevb_axes_keys).reshape(1,-1)

density_axes_keys = ["density"]
density_spec = np.array(density_axes_keys*meanb_spec.shape[1]).reshape(1,-1)

spec = np.concatenate([density_spec, meanb_spec, stddevb_spec], axis=0)

axd = fig.subplot_mosaic(spec, gridspec_kw=dict(height_ratios=[3, 2, 2]), per_subplot_kw={ak: {"projection": cp_model_rotated_pole} for ak in meanb_axes_keys + stddevb_axes_keys})

ax = axd["density"]
plot_freq_density(hist_data, target_da=cpm_pr_on_gcm, ax=ax, target_label="CPM@60km", linewidth=1)
ax.annotate("a.", xy=(0.04, 1.0), xycoords=("figure fraction", "axes fraction"), weight='bold', ha="left", va="bottom")

meanb_axes = [ axd[f'meanb {bias["label"]}'] for bias in mean_biases ]
plot_biases(mean_biases, meanb_axes, fig, transform=platecarree)
meanb_axes[0].annotate("b.", xy=(0.04, 1.0), xycoords=("figure fraction", "axes fraction"), weight='bold', ha="left", va="bottom")
meanb_axes[0].annotate("Mean", xy=(0.04, 0.5), xycoords=("figure fraction", "axes fraction"), ha="left", va="center", fontsize="medium", rotation=90)

stdb_axes = [ axd[f'stddevb {bias["label"]}'] for bias in std_biases ]
plot_biases(std_biases, stdb_axes, fig,transform=platecarree)
stdb_axes[0].annotate("c.", xy=(0.04, 1.0), xycoords=("figure fraction", "axes fraction"), weight='bold', ha="left", va="bottom")
stdb_axes[0].annotate("Std. dev.", xy=(0.04, 0.5), xycoords=("figure fraction", "axes fraction"), ha="left", va="center", fontsize="medium", rotation=90)

In [ ]:
density_axes_keys = list(map(lambda x: f"density {x}", [ hd["label"] for hd in hist_data ]))
if len(density_axes_keys) % 2 == 1:
    density_axes_keys = density_axes_keys + ["."]
density_spec = np.array(density_axes_keys).reshape(-1,2)

fig = plt.figure(figsize=(10, 5*density_spec.shape[0]))

axd = fig.subplot_mosaic(density_spec)

for hd in hist_data:
    ax = axd[f"density {hd['label']}"]
    plot_freq_density([hd], target_da=cpm_pr_on_gcm, ax=ax, target_label="CPM@60km")

In [ ]:
fig = plt.figure(layout='constrained', figsize=(2.5, 2.5))

ax = fig.subplots()

ax.plot(raw_gcm_pr.values.flat, cpm_pr_on_gcm.values.flat, linewidth=0, alpha=0.1, marker="o", markersize=0.5, color="tab:purple")
lims = [
    np.min([ax.get_xlim(), ax.get_ylim()]),
    np.max([ax.get_xlim(), ax.get_ylim()]),
]
ax.set_aspect("equal")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.plot(
    [0, 1],
    [0, 1],
    transform=ax.transAxes,
    linewidth=1,
    color="black",
    linestyle="--",
    label="Ideal",
)

ax.set_title("Gridboxes", fontsize="medium")
ax.set_xlabel(
    f"cCPM\n{xr.plot.utils.label_from_attrs(da=cpm_pr_on_gcm)}",
    fontsize="small",
)
ax.set_ylabel(
    f"GCM \n{xr.plot.utils.label_from_attrs(da=raw_gcm_pr)}",
    fontsize="small",
)

plt.show()

display(xr.corr(cpm_pr_on_gcm, raw_gcm_pr, dim=["ensemble_member", "time", "longitude", "latitude"]).rename(f"GCM CPM corr"))

In [ ]:
fig = plt.figure(layout='constrained', figsize=(2.5, 2.5))

ax = fig.subplots()

ax.plot(raw_gcm_pr.cf.mean(["X", "Y"]).values.flat, cpm_pr_on_gcm.cf.mean(["X", "Y"]).values.flat, linewidth=0, alpha=0.5, marker="o", markersize=0.5, color="tab:purple")
lims = [
    np.min([ax.get_xlim(), ax.get_ylim()]),
    np.max([ax.get_xlim(), ax.get_ylim()]),
]
ax.set_aspect("equal")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.plot(
    [0, 1],
    [0, 1],
    transform=ax.transAxes,
    linewidth=1,
    color="black",
    linestyle="--",
    label="Ideal",
)

ax.set_title("Domain mean", fontsize="medium")
ax.set_xlabel(
    f"cCPM\n{xr.plot.utils.label_from_attrs(da=cpm_pr_on_gcm)}",
    fontsize="small",
)
ax.set_ylabel(
    f"GCM \n{xr.plot.utils.label_from_attrs(da=raw_gcm_pr)}",
    fontsize="small",
)

plt.show()

display(xr.corr(cpm_pr_on_gcm.cf.mean(["X", "Y"]), raw_gcm_pr.cf.mean(["X", "Y"]), dim=["ensemble_member", "time"]).rename(f"GCM CPM domain mean corr"))

In [ ]:
fig = plt.figure(layout='constrained', figsize=(7, 2.5))

ax = fig.subplots()

# ax.plot(raw_gcm_pr.mean(axis=["X", "Y"[).flat, cpm_pr_on_gcm.values.flat, linewidth=0, alpha=0.1, marker="o", markersize=0.5, color="tab:purple")
ax.plot(range(len(raw_gcm_pr["time"])), raw_gcm_pr.isel(ensemble_member=0).cf.mean(["X", "Y"]), linewidth=0.5, label="GCM")
ax.plot(range(len(raw_gcm_pr["time"])), cpm_pr_on_gcm.isel(ensemble_member=0).cf.mean(["X", "Y"]), linewidth=0.5, label="cCPM@60km")
ax.legend()